# Response Engineering for Training-Free Alignment
---

## Overview

A key challenge in deploying language models is ensuring their outputs are helpful, accurate, and aligned with user intent, without the cost of retraining. **Training-free alignment** addresses this by intervening at inference time rather than modifying model weights.

This notebook covers **response engineering**: techniques that improve output quality by controlling *how* and *how many times* the model generates, then selecting or refining the best result.

| Part | Topic |
|---|---|
| **Part 1** | Static Response Engineering |
| **Part 2** | Iterative Response Engineering |

**Part 1** covers static strategies: Best-of-N sampling (with and without a reward model) and self-consistency voting, each makes one generation pass and applies a fixed selection rule. **Part 2** covers iterative techniques that run a feedback loop, generating, evaluating, and refining over multiple steps, including Reflexion and Mixture-of-Agents.

**Runtime Requirement:** This notebook requires an L4 GPU or better and will not run on T4. In Colab go to `Runtime → Change runtime type → L4 GPU`.

---
## Installation

In [ ]:
import os

!uv pip install --no-cache-dir -q "transformers<4.54.0"
!uv pip install --no-cache-dir -q "vllm==0.9.0"
!uv pip install --no-cache-dir -q accelerate sentencepiece

print("Done. Restarting kernel...")
os.kill(os.getpid(), 9)

---
## Setup

We load the generator model via vLLM at reduced memory utilization to leave headroom for the reward model.

In [1]:
import os
os.environ["VLLM_USE_V1"] = "0"

import torch
import vllm
import transformers
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

print(f"vLLM:         {vllm.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

INFO 03-10 09:54:47 [__init__.py:243] Automatically detected platform cuda.
vLLM:         0.9.0
Transformers: 4.53.3
CUDA available: True


In [2]:
GENERATOR_MODEL = "unsloth/Qwen3-1.7B"
RM_MODEL        = "Skywork/Skywork-Reward-V2-Qwen3-1.7B"

# Load generator via vLLM at reduced utilization to share GPU with RM
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
llm = LLM(model=GENERATOR_MODEL, gpu_memory_utilization=0.5)

print("Generator model ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

INFO 03-10 09:55:39 [weight_utils.py:307] Time spent downloading weights for unsloth/Qwen3-1.7B: 10.594255 seconds
INFO 03-10 09:55:39 [weight_utils.py:344] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-10 09:55:40 [default_loader.py:280] Loading weights took 1.03 seconds
INFO 03-10 09:55:40 [model_runner.py:1202] Model loading took 3.2152 GiB and 12.044618 seconds
INFO 03-10 09:55:42 [worker.py:291] Memory profiling takes 0.92 seconds
INFO 03-10 09:55:42 [worker.py:291] the current vLLM instance can use total_gpu_memory (22.03GiB) x gpu_memory_utilization (0.50) = 11.02GiB
INFO 03-10 09:55:42 [worker.py:291] model weights take 3.22GiB; non_torch_memory takes 0.04GiB; PyTorch activation peak memory takes 1.39GiB; the rest of the memory reserved for KV Cache is 6.36GiB.
INFO 03-10 09:55:42 [executor_base.py:112] # cuda blocks: 3723, # CPU blocks: 2340
INFO 03-10 09:55:42 [executor_base.py:117] Maximum concurrency for 40960 tokens per request: 1.45x
INFO 03-10 09:55:45 [model_runner.py:1512] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in t

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 03-10 09:56:18 [model_runner.py:1670] Graph capturing finished in 33 secs, took 0.21 GiB
INFO 03-10 09:56:18 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 37.78 seconds
Generator model ready.


In [3]:
# Load reward model via HuggingFace (stays in torch, no vLLM overhead)
from transformers import AutoModelForSequenceClassification

rm_tokenizer = AutoTokenizer.from_pretrained(RM_MODEL)
rm_model = AutoModelForSequenceClassification.from_pretrained(
    RM_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
rm_model.eval()

print("Reward model ready.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/500 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Reward model ready.


In [4]:
def generate(prompt, extra_params=None, logits_processors=None):
    """Generate a single response."""
    messages = [{"role": "user", "content": prompt}]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    params = dict(n=1, temperature=0.8, max_tokens=1024)
    if extra_params:
        params.update(extra_params)
    if logits_processors:
        params["logits_processors"] = logits_processors
    output = llm.generate([rendered], SamplingParams(**params), use_tqdm=False)
    return output[0].outputs[0].text.strip()


def generate_n(prompt, n=5, temperature=0.8, max_tokens=1024):
    """Generate N responses in a single vLLM call using the n parameter."""
    messages = [{"role": "user", "content": prompt}]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    params = SamplingParams(n=n, temperature=temperature, max_tokens=max_tokens)
    output = llm.generate([rendered], params, use_tqdm=False)
    return [o.text.strip() for o in output[0].outputs]


def score_response(prompt, response):
    """Score a prompt-response pair using the reward model."""
    conversation = [
        {"role": "user",      "content": prompt},
        {"role": "assistant", "content": response},
    ]
    input_text = rm_tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False
    )
    inputs = rm_tokenizer(input_text, return_tensors="pt").to(rm_model.device)
    with torch.no_grad():
        score = rm_model(**inputs).logits[0][0].item()
    return score


print("Helpers ready.")

Helpers ready.


In [5]:
def section(title):
    """Print a section header."""
    print()
    print("=" * 62)
    print(f"  {title}")
    print("=" * 62)

def divider():
    print("-" * 62)

def show_response(label, text, truncate=None):
    """Print a labeled response with optional truncation."""
    print(f"[{label}]")
    body = text[:truncate] + "..." if truncate and len(text) > truncate else text
    print(body)
    print()

print("Pretty-print helpers ready.")

Pretty-print helpers ready.


---
# Part 1: Static Response Engineering

Static response engineering generates multiple candidates in a single pass and applies a fixed, human-designed rule to select the best one. There is no feedback loop, the model generates once and the selection strategy decides the winner.

---
## 1.1 Best-of-N: vLLM Built-in

**Generate N candidates and return the one with the highest cumulative log-probability.**

vLLM's `best_of` parameter handles this natively, it samples `best_of` sequences internally and returns the single highest-scoring one based on the model's own likelihood, requiring no external scorer.

This is the simplest form of Best-of-N: the model acts as its own judge via log-probability.

In [6]:
# prompt = (
#     "A bottle and a cork cost $1.10. The bottle costs $1 more than the cork. "
#     "How many cents does the cork cost? "
#     "Show your working, then put only the numeric final answer in \\boxed{} at the end."
# )

# prompt = (
#     "A doctor gives you 3 pills and tells you to take one every half hour. "
#     "How many minutes will the pills last? "
#     "Show your working, then put the final answer in \\boxed{} at the end."
# )

# prompt = (
#     "There are 3 apples. You take away 2. How many apples do you have?"
#     "Show your working, then put the final answer in \\boxed{} at the end."
# )

prompt = (
    "A snail climbs 3m up a 10m wall each day but slips back 2m each night. "
    "How many days to reach the top? "
    "Show your working, then put the final answer in \\boxed{} at the end."
)

In [7]:
# Without Best-of-N: single sample
single = generate(prompt, extra_params={"n": 1, "temperature": 0.8})
section("Single Sample")
show_response("output", single)



  Single Sample
[output]
We are told:

- The **snail climbs 3 meters** up a **10-meter wall** each **day**.
- Then, it **slips back 2 meters** each **night**.

We want to know **how many days** it takes the snail to reach the **top** (i.e., 10 meters).

---

### Step 1: Understand the cycle

Each **day-night cycle** consists of:

- **Climbing 3m** → **Slipping 2m**
- So, net gain per day-night cycle = $ 3 - 2 = 1 $ meter

---

### Step 2: How many full cycles are needed?

After **n full cycles**, the snail has climbed:

$$
n \times 1 \text{ meter} = n \text{ meters}
$$

We want the snail to reach at least **10 meters**.

So, we need:

$$
n \times 1 \text{ meter} \geq 10 \text{ meters}
$$

That means:

$$
n \geq 10
$$

So, after **10 full cycles**, the snail has climbed **10 meters**.

---

### Step 3: What happens on the 11th day?

On the **11th day**, the snail climbs **3 meters**.

Since the snail has already reached **10 meters**, it **doesn't slip back**.

So, it reaches the top o

In [8]:
# With Best-of-N (vLLM built-in): sample 32, return best by log-prob
best_of_n = generate(prompt, extra_params={"best_of": 32, "temperature": 0.8})
section("Best-of-32 (vLLM Built-in)")
show_response("output", best_of_n)



  Best-of-32 (vLLM Built-in)
[output]
We are given:

- The snail climbs **3 meters** up a **10-meter wall** each day.
- It slips back **2 meters** each night.

We are to find out **how many days** it takes for the snail to reach the top.

---

### Step-by-step reasoning:

Let’s consider the **net progress** each **day**:

- **Morning**: The snail climbs **3 m**.
- **Evening**: It slips back **2 m**.

So, the **net gain per day** is:

$$
3 - 2 = 1 \text{ meter per day}
$$

However, we must be careful: the snail **does not slip back** on the **last day** when it reaches the top.

---

### Key observation:

On the **last day**, the snail climbs 3 meters and **reaches the top**, **without slipping back**.

So, we need to find the **day** when the snail reaches or exceeds **10 meters**.

Let’s simulate the progress day by day:

- **Day 1**:
  - Climbs 3 → 3
  - Slips back 2 → 1 (net of 1 m)

- **Day 2**:
  - Climbs 3 → 4
  - Slips back 2 → 2 (net of 1 m)

- **Day 3**:
  - Climbs 3 → 5
  - 

---
## 1.2 Best-of-N: Reward Model Scoring

**Generate N candidates and return the one with the highest reward model score.**

Unlike the built-in approach which uses the generator's own log-probabilities, here we use a dedicated **reward model** (`Skywork-Reward-V2-Qwen3-1.7B`) trained to score responses for helpfulness and quality.

This separates generation from evaluation, the generator tries to produce diverse candidates, and the RM acts as an independent judge.

```
generate N responses → score each with RM → return highest scoring
```

In [29]:
# Generate N diverse candidates
candidates = generate_n(prompt, n=32, temperature=0.8)

section("Best-of-32 with Reward Model")
scores = []
for i, candidate in enumerate(candidates):
    score = score_response(prompt, candidate)
    scores.append(score)
    divider()
    show_response(f"Candidate {i+1}  score={score:.4f}", candidate, truncate=1024)

best_idx = scores.index(max(scores))
section(f"Best Candidate: #{best_idx+1}  score={scores[best_idx]:.4f}")
show_response("output", candidates[best_idx])



  Best-of-32 with Reward Model
--------------------------------------------------------------
[Candidate 1  score=10.7188]
We are given:

- **Height of the wall**: 10 meters  
- **Distance climbed during the day**: 3 meters  
- **Distance slipped during the night**: 2 meters  

---

### Step-by-step analysis:

Let’s track the snail’s progress day by day.

#### Day 1:
- **Climbs 3m** → Position = 3 meters  
- **Slips 2m** → Position = 3 - 2 = **1m**

#### Day 2:
- **Climbs 3m** → Position = 1 + 3 = **4m**  
- **Slips 2m** → Position = 4 - 2 = **2m**

#### Day 3:
- **Climbs 3m** → Position = 2 + 3 = **5m**  
- **Slips 2m** → Position = 5 - 2 = **3m**

#### Day 4:
- **Climbs 3m** → Position = 3 + 3 = **6m**  
- **Slips 2m** → Position = 6 - 2 = **4m**

#### Day 5:
- **Climbs 3m** → Position = 4 + 3 = **7m**  
- **Slips 2m** → Position = 7 - 2 = **5m**

#### Day 6:
- **Climbs 3m** → Position = 5 + 3 = **8m**  
- **Slips 2m** → Position = 8 - 2 = **6m**

#### Day 7:
- **Climbs 3m** → Posit

---
## 1.3 Self-Consistency

**Sample N reasoning paths and select the answer by majority vote.**

Proposed by Wang et al. (2022), self-consistency is particularly effective for reasoning tasks. The intuition is that while the model may take different reasoning paths, correct answers tend to converge while incorrect ones tend to diverge.

```
generate N responses → extract final answers → majority vote → return most common
```

This works best for questions with a discrete, extractable answer (math, factual QA, multiple choice).

In [12]:
from collections import Counter
import re

def extract_boxed(text):
    """Extract the value inside \\boxed{} and clean latex artifacts."""
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if not match:
        return text.strip()
    value = match.group(1).strip()
    # Remove latex text annotations, both closed \text{...} and unclosed \text{...
    value = re.sub(r'\\text\{[^}]*\}?', '', value)
    # Remove other latex commands e.g. \approx, \cdot
    value = re.sub(r'\\[a-zA-Z]+', '', value)
    # Remove commas from numbers e.g. 1,440 -> 1440
    value = value.replace(',', '')
    return value.strip()

def majority_vote(answers):
    """Return the most common answer from a list."""
    normalized = [a.strip().lower().rstrip(".,!?") for a in answers]
    count = Counter(normalized)
    most_common = count.most_common(1)[0]
    for original, norm in zip(answers, normalized):
        if norm == most_common[0]:
            return original, most_common[1]

print("majority_vote() and extract_boxed() defined.")

majority_vote() and extract_boxed() defined.


In [13]:
responses = generate_n(prompt, n=32, temperature=0.8, max_tokens=1024)

section("Sampled Responses")
for i, r in enumerate(responses, 1):
    divider()
    show_response(f"Response {i:2d}", r)

extracted = [extract_boxed(r) for r in responses]
section("Extracted Answers")
for i, e in enumerate(extracted, 1):
    print(f"  {i:2d}. {e}")

winner, vote_count = majority_vote(extracted)
section(f"Majority Vote: {winner}  ({vote_count}/32 responses)")



  Sampled Responses
--------------------------------------------------------------
[Response  1]
We are given:

- The snail climbs **3 meters** up a **10-meter** wall each day.
- Then it slips back **2 meters** each night.

We need to find **how many days** it takes the snail to reach the **top of the wall**.

---

### Step 1: Understand the net gain per day

Each day, the snail climbs **3 meters**, then slips back **2 meters** during the night.

So the **net gain per day** is:

$$
3 - 2 = 1 \text{ meter per day}
$$

---

### Step 2: How many days does it take to reach the top?

On the **last day**, the snail climbs **3 meters**, and if it reaches or exceeds the top, it doesn't slip back.

So, we need to determine how many full days of climbing and slipping are needed before the snail reaches the top.

Let’s assume the snail reaches the top on the **$n$-th day**.

On the **$n-1$-th day**, the snail has a net gain of **1 meter**, so it’s at:

$$
(n - 1) \times 1 = n - 1 \text{ meters}


---
# Part 2: Iterative Response Engineering

Iterative response engineering runs a **feedback loop** rather than a single post-processing pass:

1. Generate a response
2. Evaluate it (model, verifier, or environment)
3. Refine based on feedback
4. Repeat until a quality threshold is met

This is the key distinction from static response engineering: instead of selecting once from a fixed set of candidates, the system iteratively improves the response over multiple steps.

---
## 2.1 Reflexion

**Iteratively refine an answer through a self-critique feedback loop.**

Reflexion (Shinn et al., 2023) prompts the model to evaluate its own output, identify weaknesses, and produce an improved version. This loop runs for a fixed number of rounds, with each round feeding the previous answer back as input.

```
generate answer
  → evaluate: critique the answer
    → refine: revise based on critique
      → evaluate again ... (repeat for N rounds)
```

The same model acts as both generator and critic. Each round is a full generate-evaluate-refine cycle, making this iterative rather than a one-shot selection.

In [17]:
def reflexion(prompt, rounds=3, max_tokens=1024):
    """
    Iteratively refine an answer through self-critique.

    Args:
        prompt:     The original user question.
        rounds:     Number of critique-and-revise cycles.
        max_tokens: Max tokens per generation call.

    Returns:
        List of (answer, critique) tuples for each round, plus the final answer.
    """
    history = []

    answer = generate(prompt, extra_params={"temperature": 0.8, "max_tokens": max_tokens})
    section("Initial Answer")
    show_response("output", answer)

    for round_num in range(1, rounds + 1):
        critique_prompt = (
            f"Here is a question and an answer. Check the question carefully to determine the actual requirement. "
            f"Analyze every single step of the answer carefully and identify any errors, omissions, or misunderstandings. "
            f"Be specific, and do not invent issues.\n\n"
            f"Question: {prompt}\n\n"
            f"Answer: {answer}"
        )
        critique = generate(critique_prompt, extra_params={"temperature": 0.8, "max_tokens": max_tokens})

        revise_prompt = (
            f"Here is a question, an answer, and a critique of that answer. "
            f"Write an improved answer that addresses the critique.\n\n"
            f"Question: {prompt}\n\n"
            f"Previous answer: {answer}\n\n"
            f"Critique: {critique}\n\n"
            f"Improved answer:"
        )
        revised = generate(revise_prompt, extra_params={"temperature": 0.8, "max_tokens": max_tokens})

        history.append((answer, critique))

        section(f"Round {round_num}")
        show_response("critique", critique)
        show_response("revised", revised)

        answer = revised

    return history, answer


print("reflexion() defined.")


reflexion() defined.


In [18]:
history, final_answer = reflexion(prompt, rounds=3)

section("Final Answer after 3 Rounds of Reflexion")
show_response("output", final_answer)



  Initial Answer
[output]
We are given the following:

- **Daily climb**: The snail climbs **3 meters** up the wall.
- **Night slip**: The snail slips back **2 meters** during the night.
- The wall is **10 meters** tall.

---

### Step 1: Net gain per day

Each day, the snail climbs **3 meters**, then slips back **2 meters** at night.

So the **net gain per day** is:

$$
3 - 2 = 1 \text{ meter per day}
$$

---

### Step 2: How many days to reach the top

We want to find how many days it takes for the snail to **reach or exceed** the 10-meter wall.

Let’s simulate the process:

- **Day 1**:
  - Climbs 3 meters → 3 meters
  - Slips back 2 meters → 1 meter

- **Day 2**:
  - Climbs 3 meters → 4 meters
  - Slips back 2 meters → 2 meters

- **Day 3**:
  - Climbs 3 meters → 5 meters
  - Slips back 2 meters → 3 meters

- **Day 4**:
  - Climbs 3 meters → 6 meters
  - Slips back 2 meters → 4 meters

- **Day 5**:
  - Climbs 3 meters → 7 meters
  - Slips back 2 meters → 5 meters

- **Day 6**:
  -

---
## 2.2 Mixture-of-Agents

**Generate N independent responses, then feed all of them back into the same model to synthesize a final answer.**

Inspired by Wang et al. (2024), Mixture-of-Agents (MoA) treats multiple model outputs as a panel of independent perspectives. Unlike Best-of-N which selects one candidate, MoA uses a second model call to synthesize all responses, creating a two-step loop of generation followed by aggregation.

```
generate N independent responses (vLLM n parameter)
  → evaluate and synthesize: feed all N responses back to the model
    → model produces a final consolidated answer
```

Using vLLM's `n` parameter means all N responses are generated in a single batch, efficient and consistent.

In [20]:
def mixture_of_agents(prompt, n=5, temperature=0.8, max_tokens=1024):
    """
    Generate N responses and aggregate them into a final answer.

    Args:
        prompt:      The original user question.
        n:           Number of agent responses to generate.
        temperature: Sampling temperature for the agent responses.
        max_tokens:  Max tokens per generation step.

    Returns:
        agents_responses: List of N individual responses.
        final_answer:     Aggregated final answer.
    """
    # Step 1: Generate N independent responses in one vLLM call
    agent_responses = generate_n(prompt, n=n, temperature=temperature, max_tokens=max_tokens)

    print(f"Generated {n} agent responses:")
    for i, r in enumerate(agent_responses, 1):
        print(f"\n--- Agent {i} ---")
        print(r)

    # Step 2: Aggregate using the same model
    responses_text = "\n\n".join(
        f"Response {i+1}:\n{r}" for i, r in enumerate(agent_responses)
    )

    aggregation_prompt = (
        f"Below are {n} independent responses to the following question.\n\n"
        f"Question: {prompt}\n\n"
        f"{responses_text}\n\n"
        f"Synthesize these responses into a single, comprehensive, and accurate final answer. "
        f"Resolve any contradictions and include the best points from each response."
    )

    final_answer = generate(
        aggregation_prompt,
        extra_params={"temperature": 0.8, "max_tokens": max_tokens}
    )

    return agent_responses, final_answer


print("mixture_of_agents() defined.")

mixture_of_agents() defined.


In [21]:
agent_responses, final_answer = mixture_of_agents(prompt, n=16)

section("Aggregated Final Answer (Mixture-of-Agents)")
show_response("output", final_answer)


Generated 16 agent responses:

--- Agent 1 ---
We are given:

- The snail climbs **3 meters** up a **10-meter wall** each day.
- Then it slips back **2 meters** each night.

We want to know **how many days** it takes for the snail to reach the top.

---

### Step 1: Understand the net progress each day

Each day:
- The snail climbs **3 meters**.
- Then slips back **2 meters**.

So the **net progress per day** is:

$$
3 - 2 = 1 \text{ meter per day}
$$

But we must be careful: **on the day it reaches the top, it doesn't slip back**.

---

### Step 2: How many days does it take to reach or exceed 10 meters?

Let’s simulate the climbing and slipping.

#### Day 1:
- Climbs 3m → Now at 3m.
- Slips back 2m → Now at 1m.

#### Day 2:
- Climbs 3m → Now at 4m.
- Slips back 2m → Now at 2m.

#### Day 3:
- Climbs 3m → Now at 5m.
- Slips back 2m → Now at 3m.

#### Day 4:
- Climbs 3m → Now at 6m.
- Slips back 2m → Now at 4m.

#### Day 5:
- Climbs 3m → Now at 7m.
- Slips back 2m → Now at 5m.

#### Day

---
## Summary

| Technique | Part | Loop? | Selection strategy | Alignment signal |
|---|---|---|---|---|
| Best-of-N (built-in) | Static | No | Highest log-probability | Model's own likelihood |
| Best-of-N (RM) | Static | No | Highest reward score | External reward model |
| Self-consistency | Static | No | Majority vote | Agreement across samples |
| Reflexion | Iterative | Yes | Self-critique and revision | Model as evaluator |
| Mixture-of-Agents | Iterative | Yes | Model synthesis | Model as aggregator |

All five techniques are **training-free**, they improve alignment purely through inference-time computation, with no gradient updates or weight changes.

---
## Resources

**Self-Consistency Improves Chain of Thought Reasoning in Language Models**
Wang et al. (2022) - the original self-consistency paper.
https://arxiv.org/abs/2203.11171

**Reflexion: Language Agents with Verbal Reinforcement Learning**
Shinn et al. (2023) - the original reflexion paper.
https://arxiv.org/abs/2303.11366

**Mixture-of-Agents Enhances Large Language Model Capabilities**
Wang et al. (2024) - the original MoA paper.
https://arxiv.org/abs/2406.04692

**Skywork-Reward-V2**
The reward model used in this notebook.
https://huggingface.co/Skywork/Skywork-Reward-V2-Qwen3-0.6B

**vLLM Documentation**
https://docs.vllm.ai